
# Práctica NLP – Sentiment Analysis con reviews de Amazon

### Máster en Deep Learning UPM - Curso 2025/26


## 1. Carga del dataset

**Usaremos un dataset para análisis de sentimiento en reviews de Amazon**


Comenzamos añadiendo la conexión directa a la ubicación donde se situa el dataset a analizar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).



## 2. Exploración del dataset

**Analiza el dataset y los pasos de normalización que puedan ser necesarios**

La practica empieza obteniendo las primeras referencias del dataset proporcionado, para ello, se realiza un analisis mas visual mostrando en primera estancia los datos contenidos en el panda.

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/My Drive/UPM/NLP/dataset_practica.csv')

print(df)

                                             review_body language  \
0      Très contente, j'en ai acheté 2 car mes 2 garç...       fr   
1      サイズはピッタリフィットしたけど、実際に使ってみると頭を傾けただけでよく滑り落ちる。 使用感...       ja   
2      livré dans les délais semble solide a voir a l...       fr   
3        Une des deux boîtes des chargeurs était cassée.       fr   
4                       si ha cumplido mis espectativas.       es   
...                                                  ...      ...   
23973     Meine Box ist immer sicher und gut verpackt. 👍       de   
23974                              自我暗示，自我激励，关于如何分析问题的书。       zh   
23975  Schnelle Lieferung, ausgepackt und so ein edle...       de   
23976                            上脚没有那么好看，因为很单薄，所以导致有点偏大       zh   
23977  到货验证了下，照着网上的防伪验证了下是假货。本以为亚马逊是所有电商里最靠谱的，没想到也卖假货...       zh   

      sentiment_label        product_category  
0            positive                      pc  
1            negative             electronics  
2            positive      

Como se puede observar el dataset esta formado por las siguientes columnas:
* **Review_body**: El comentario añadido al producto.
* **language**: El idioma en el que se esta escribiendo el comentario.
* **Sentiment_label**: La etiqueta que indica si el comentario es positivo o negativo.
* **Product_category**: Categoria del producto.

Podemos deducir a partir del dataset que el objetivo de la practica es predecir si la review es negativa o positiva a partir de la review del cliente.

Otro punto a tener en cuenta, es que existen elementos en el texto que pueden interceder en el entrenamiento del modelo, como por ejemplo:
* Emojis
* Acentos
* Errores gramaticales
* Caracteres especiales

Estos elementos habria que eliminarlos para facilitar el trabajo de entrenamiento del modelo quedando el comentario lo mas limpio posible, esta limpieza se lleva a cabo en el siguiente apartado de la práctica, mientras tanto seguimos sacando información del dataset proporcionado.

In [ ]:
lenguajes = df['language'].unique()
print(lenguajes)

['fr' 'ja' 'es' 'de' 'en' 'zh']


Agrupamos las reviews para observar el conjunto de idiomas utilizados en el dataset, en este caso lo encontramos en formato ISO:
* **fr**: Francés
* **ja**: Japonés
* **es**: Español
* **de**: Neerlandés
* **en**: Inglés
* **zh**: Chino



## 3. Presencia de diferentes idiomas

### Debes justificar una de las siguientes opciones (u otra propuesta por ti):

- Usar **un solo idioma**
- Usar **varios idiomas combinados**
- Traducir todos los textos a un idioma común
- Entrenar **modelos separados por idioma**
- Filtrar idiomas con pocos ejemplos

**Describe y justifica tu decisión antes de continuar.**


Antes de tomar una decisión que determine el camino que se va a llevar a cabo en el proyecto, vamos a realizar una serie de comprobaciones que van a permitir tomar una decisión.

Primero se realizará una comprobación de cuantas reviews hay en cada idioma

In [ ]:
lenguajes_count = df.groupby('language').count()
print(lenguajes_count)

          review_body  sentiment_label  product_category
language                                                
de               3977             3977              3977
en               4033             4033              4033
es               3979             3979              3979
fr               3989             3989              3989
ja               3995             3995              3995
zh               4005             4005              4005


El numero de reviews en cada idioma es parecido, todos en torno a los 4000 comentarios, con esta información no podemos tomar ninguna decisión de peso para descartar alguna de las opciones anteriormente propuestas.


Pero se puede sacar ciertas conclusiones:
* El chino y el inglés son los idiomas con mas reseñas.
* El aleman es el idioma con menos reviews.



En este proyecto se ha propuesto acotar a un solo idioma, el idioma elegido será el inglés y a parte de las razones obvias, como que el inglés se trata del lenguaje mas utilizado a nivel mundial, existen otros motivos de peso por lo que se ha elegido esta propuesta:

* La utilización de modelos multilingues pueden traer un peor rendimiento que el entrenamiento con un solo idioma, priorizando el rendimiento y sacrificando el multilenguaje.

* El idioma con mas bibliotecas y recursos en NLP se trata del ingles, los recursos linguisticos esta desbalanceados, es decir existe una mayor disponibilidad de recursos entre unos y otros idiomas como es de esperar.




## 4. Preprocesado con expresiones regulares

Aplica y justifica los pasos de normalización necesarios al dataset

Observando las reviews del dataset, se llega a la conclusión de que los datos no estan normalizados y esto puede provocar problemas a la hora de entrenar los modelos o incluso en la transformación en vectores.

Teniendo en cuenta que se trata de opiniones proporcionadas por usuarios los datos tienen desde emojis hasta caracteres raros, obligando a una normalización de todas las reviews. Estas son las reglas aplicadas:

* Eliminación de mayusculas
* Eliminación de números
* Eliminación de signos de puntuación
* Eliminación de espacios y tabulaciones
* Eliminación de emojis.

In [ ]:
import re

def dataset_clean(text):
  clean_text = dataset_clean_expresions(text)

  #Eliminacion de stop words
  clean_text = delete_stop_words(clean_text)
  return clean_text


def dataset_clean_expresions(text):
    #Minusculas
    clean_text = text.lower()
    #ELiminar numeros
    clean_text = re.sub(r'\d+', '', clean_text)
    #Eliminar simbolos puntuacion
    clean_text = re.sub(r'[^\w\s]', '', clean_text)
    #Eliminar espacios multiples
    clean_text = re.sub(r'\s+', ' ', clean_text)
    #Eliminar todos los emojis
    clean_text = re.sub(r'[^\x00-\x7F]+', '', clean_text)
    return clean_text


Por ultimo se hace uso de una libreria de stop words en ingles, las stop words se tratan de un conjunto de preposiciones que no aportan gran significado al computo total y que tienen un uso frecuente en las oraciones. Al no aportar un valor real para la posterior vectorizacion podemos desecharlas.

In [ ]:
from nltk.corpus import stopwords
import nltk

#Descarga stopwords en ingles
nltk.download('stopwords')

def delete_stop_words(clean_text):
  stop_words = set(stopwords.words('english'))

  clean_text = ' '.join([word for word in clean_text.split() if word not in stop_words])
  return clean_text

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



## 5. Entrenamiento y evaluación

- Prepara una representación basada en bagofwords y otra usando word2vec para conseguir un vector por cada review
- Entrena distintos clasificadores de sklearn y evalúa los resultados



Como se ha comentado anteriormente se trabajará directamente solo con los datos realizados en ingles, por lo tanto preparamos el dataset para trabajar en este idioma, este dataset será utilizado para la vectorización y en el posterior entrenamiento de los modelos.

In [ ]:
en_df = df.loc[df['language'] == 'en'].reset_index(drop=True)
print(en_df)

                                            review_body language  \
0     This made to fit all seems to fit none well. F...       en   
1     This would make a great travel mirror. However...       en   
2     The color balance and quality of the image are...       en   
3     I had to purchase new headphones for my colleg...       en   
4     This was a cute set. My daughter loved it. She...       en   
...                                                 ...      ...   
4028  Bought 2; they both died two after opening the...       en   
4029  Awesome product. As seen in my picture, you ca...       en   
4030                       Color more gold than yellow.       en   
4031  A friend told me about this little gadget, I s...       en   
4032  This well written story was incredible.The cha...       en   

     sentiment_label        product_category  
0           negative              automotive  
1           negative                  beauty  
2           positive                      

Como se verá mas adelante se realizará una normalización diferente para cada propuesta de vectorización, adaptandonos a los embeddings que vamos a utilizar.



## BAG OF WORDS
Se trata de un algoritmo que permite transformar el texto a vectores contando la frecuencia de aparición de las palabras en el dataset, no tiene en cuenta el orden de aparición de las palabras. Por esta razon es importante realizar una buena normalización del dataset, facilitando la detección de palabras de utilidad y eliminarndo stop_words que no aportan ningún tipo de información.

Empezaremos aplicando las reglas de normalizacion que hemos definido en el apartado anterior, generando una nueva columna llamada **clean_review** con la cual se va a llevar a cabo todos y cada uno de los procesos.

In [ ]:
en_df['clean_review'] = en_df['review_body'].apply(dataset_clean)

en_df['clean_review']

,clean_review
0,made fit seems fit none well followed videos s...
1,would make great travel mirror however unfortu...
2,color balance quality image great love
3,purchase new headphones college piano course i...
4,cute set daughter loved shes years old size pr...
...,...
4028,bought died two opening tried recharge batteri...
4029,awesome product seen picture tell stopped phot...
4030,color gold yellow
4031,friend told little gadget sew lot make bias ta...


A continuación, veremos como vectoriza **bag of words**, generando un conjunto de todas las palabras únicas y luego generando un vector de frecuencias de dichas palabras por cada reseña.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

bag_of_words = vectorizer.fit_transform(en_df['clean_review'])

bag_of_words.shape

(4033, 8750)

Pasamos a realizar en la partición del dataset en train y test, aplicaremos una distribución: 80/20.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

X_text = en_df['clean_review']
y = en_df['sentiment_label']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2
)

vectorizer = CountVectorizer()

X_train = vectorizer.fit_transform(X_train_text)
X_test  = vectorizer.transform(X_test_text)

Los modelos elegidos para el entrenamiento son los siguientes:
* Naive Bayes
* Logistic Regression
* Random Forest

Todos pertenecientes a sklearn.

In [ ]:
from sklearn.metrics import classification_report

modelos = {
    "NaiveBayes": MultinomialNB(),
    "LogisticRegression": LogisticRegression(max_iter=2000),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
}

lista_metricas = []

for name, model in modelos.items():
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

  rep_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

  df_bow = pd.DataFrame(rep_dict).T
  df_bow["model"] = name
  df_bow["label"] = df_bow.index
  df_bow = df_bow.reset_index(drop=True)

  lista_metricas.append(df_bow)

df_results_bow = pd.concat(lista_metricas)
df_results_bow

,precision,recall,f1-score,support,model,label
0,0.828715,0.788969,0.808354,417.000000,NaiveBayes,negative
1,0.785366,0.825641,0.805000,390.000000,NaiveBayes,positive
2,0.806691,0.806691,0.806691,0.806691,NaiveBayes,accuracy
3,0.807041,0.807305,0.806677,807.000000,NaiveBayes,macro avg
4,0.807766,0.806691,0.806733,807.000000,NaiveBayes,weighted avg
0,0.828205,0.774580,0.800496,417.000000,LogisticRegression,negative
1,0.774580,0.828205,0.800496,390.000000,LogisticRegression,positive
2,0.800496,0.800496,0.800496,0.800496,LogisticRegression,accuracy
3,0.801393,0.801393,0.800496,807.000000,LogisticRegression,macro avg
4,0.802290,0.800496,0.800496,807.000000,LogisticRegression,weighted avg


##Word2Vec
Se trata de un embedding, funciones que permiten representar lenguaje natural a vectores, facilitando asi el procesamiento de los modelos para poder capturar relaciones semanticas entre las diferentes palabras segun la similitud del significado de cada una.



In [ ]:
!pip -q install gensim

In [ ]:
from gensim.models import word2vec

Mientras que en el modelo de Bag Of Words se trabajaba con la frecuencia de palabras, en el words2vec como se ha comentado capturamos las relaciones semanticas por lo tanto daremos una transformación diferente al dataset, manteniendo las stop words, ya que en este caso si que nos pueden aportar informacion extra.

Realizamos un preprocesado especifico para W2V, en este preprocesado existen reviews que quedan vacias, por lo tanto se identifican y se eliminan antes de comenzar con el entrenamiento del modelo.

In [ ]:
import re

def dataset_clean_W2V(text):
  return dataset_clean_expresions(text)

In [ ]:
en_df['clean_review_w2v'] = en_df['review_body'].apply(dataset_clean_W2V)


en_df = en_df.dropna(subset=['clean_review_w2v']).reset_index(drop=True)

en_df['clean_review_w2v']

,clean_review_w2v
0,this made to fit all seems to fit none well fo...
1,this would make a great travel mirror however ...
2,the color balance and quality of the image are...
3,i had to purchase new headphones for my colleg...
4,this was a cute set my daughter loved it shes ...
...,...
4028,bought they both died two after opening them t...
4029,awesome product as seen in my picture you can ...
4030,color more gold than yellow
4031,a friend told me about this little gadget i se...


Una vez se han normalizado las reviews, empieza el proceso de tokenizacion de las reseñas en el que cada una se divide en un listado de palabras. Proceso fundamental ya que W2V no trabaja directamente con texto sino con un conjunto de palabras estableciendo relaciones semanticas a partir del contexto de las reseñas.

In [ ]:
reviews_list = en_df['clean_review_w2v'].apply(lambda x: x.split()).tolist()
reviews_list

[['this',
  'made',
  'to',
  'fit',
  'all',
  'seems',
  'to',
  'fit',
  'none',
  'well',
  'followed',
  'videos',
  'and',
  'still',
  'the',
  'blades',
  'do',
  'not',
  'stay',
  'attached',
  'tried',
  'on',
  'another',
  'vehicle',
  'with',
  'adapter',
  'and',
  'they',
  'stay',
  'on',
  'but',
  'are',
  'loose',
  'do',
  'not',
  'know',
  'how',
  'well',
  'they',
  'hold',
  'up',
  'because',
  'i',
  'could',
  'not',
  'use',
  'them'],
 ['this',
  'would',
  'make',
  'a',
  'great',
  'travel',
  'mirror',
  'however',
  'unfortunately',
  'the',
  'lights',
  'were',
  'so',
  'bright',
  'i',
  'couldnt',
  'see',
  'my',
  'face',
  'well',
  'about',
  'to',
  'apply',
  'makeup',
  'the',
  'side',
  'mirrors',
  'were',
  'too',
  'small',
  'too'],
 ['the',
  'color',
  'balance',
  'and',
  'quality',
  'of',
  'the',
  'image',
  'are',
  'great',
  'love',
  'it'],
 ['i',
  'had',
  'to',
  'purchase',
  'new',
  'headphones',
  'for',
  'my',
 

Ahora tenemos que vectorizar cada una de las palabras separadas en la tokenizacion por cada lista y hacemos la media de cada review, obteniendo un significado global de cada una.

In [ ]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=reviews_list,
    vector_size=100,
    window=5,
    min_count=2
)

In [ ]:
import numpy as np

def vectorize_review(vector, model):
  vectors = [model.wv[word] for word in vector if word in model.wv]
  if len(vectors) == 0:
    return np.zeros(model.vector_size)
  return np.mean(vectors, axis=0)


X_vect = np.array([vectorize_review(vector, w2v_model) for vector in reviews_list])

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

y = en_df['sentiment_label']

X_train, X_test, y_train, y_test = train_test_split(X_vect, y, test_size=0.2)

modelos = {
    "LogisticRegression": LogisticRegression(max_iter=2000),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
}

lista_metricas = []

for name, model in modelos.items():
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

  rep_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

  df_w2v = pd.DataFrame(rep_dict).T
  df_w2v["model"] = name
  df_w2v["label"] = df_w2v.index
  df_w2v = df_w2v.reset_index(drop=True)

  lista_metricas.append(df_w2v)

df_results_w2v = pd.concat(lista_metricas)
df_results_w2v


,precision,recall,f1-score,support,model,label
0,0.638051,0.684080,0.660264,402.000000,LogisticRegression,negative
1,0.662234,0.614815,0.637644,405.000000,LogisticRegression,positive
2,0.649318,0.649318,0.649318,0.649318,LogisticRegression,accuracy
3,0.650143,0.649447,0.648954,807.000000,LogisticRegression,macro avg
4,0.650187,0.649318,0.648912,807.000000,LogisticRegression,weighted avg
0,0.682620,0.674129,0.678348,402.000000,RandomForest,negative
1,0.680488,0.688889,0.684663,405.000000,RandomForest,positive
2,0.681537,0.681537,0.681537,0.681537,RandomForest,accuracy
3,0.681554,0.681509,0.681505,807.000000,RandomForest,macro avg
4,0.681550,0.681537,0.681517,807.000000,RandomForest,weighted avg


# 6. Conclusiones
Vistos los resultados de los modelos, pasamos a la comparación de todas las metricas para comprobar que modelo ha dado mejor resultado, en este caso se comparan los modelos comunes a ambas propuestas:

* Logistic Regression
* Random Forest

Mostramos las tablas de metricas a continuación

In [ ]:
df_results_bow

,precision,recall,f1-score,support,model,label
0,0.828715,0.788969,0.808354,417.000000,NaiveBayes,negative
1,0.785366,0.825641,0.805000,390.000000,NaiveBayes,positive
2,0.806691,0.806691,0.806691,0.806691,NaiveBayes,accuracy
3,0.807041,0.807305,0.806677,807.000000,NaiveBayes,macro avg
4,0.807766,0.806691,0.806733,807.000000,NaiveBayes,weighted avg
0,0.828205,0.774580,0.800496,417.000000,LogisticRegression,negative
1,0.774580,0.828205,0.800496,390.000000,LogisticRegression,positive
2,0.800496,0.800496,0.800496,0.800496,LogisticRegression,accuracy
3,0.801393,0.801393,0.800496,807.000000,LogisticRegression,macro avg
4,0.802290,0.800496,0.800496,807.000000,LogisticRegression,weighted avg


In [ ]:
df_results_w2v

,precision,recall,f1-score,support,model,label
0,0.638051,0.684080,0.660264,402.000000,LogisticRegression,negative
1,0.662234,0.614815,0.637644,405.000000,LogisticRegression,positive
2,0.649318,0.649318,0.649318,0.649318,LogisticRegression,accuracy
3,0.650143,0.649447,0.648954,807.000000,LogisticRegression,macro avg
4,0.650187,0.649318,0.648912,807.000000,LogisticRegression,weighted avg
0,0.682620,0.674129,0.678348,402.000000,RandomForest,negative
1,0.680488,0.688889,0.684663,405.000000,RandomForest,positive
2,0.681537,0.681537,0.681537,0.681537,RandomForest,accuracy
3,0.681554,0.681509,0.681505,807.000000,RandomForest,macro avg
4,0.681550,0.681537,0.681517,807.000000,RandomForest,weighted avg


Como podemos comprobar los resultados no son los esperados en cuanto a Word2Vec se refiere, ya que ha conseguido tener unas metricas 20% inferiores al rendimiento de Bags Of Words, esto puede deberse a que necesita datasets mas grandes para poder capturar mas estructura contextual y relaciones semanticas.

En cuanto a Bag Of Words, ha mostrado un muy buen rendimiento en todas las metricas. Llegando a la conclusión que para este dataset en concreto y siguiendo esta practica de clasificación, resulta mas eficaz la frecuencia de los caracteres, dada por Bags of words, que el uso de relaciones semanticas generadas a partir de Word2Vec